In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
from workflow.scripts.utils import global_avg
import pandas as pd
import matplotlib as mpl
from workflow.scripts.plotting_tools import get_model_colordict
from scipy.stats import linregress
%matplotlib inline

In [ ]:
vars_cld = ['clivi','pr']
vars_diag = ['radatm','radatmcs','concdust','abs550aer']

ds_exp_cld = {p.split("_")[-2]: xr.open_dataset(p).isel(time=slice(1,None))[vars_cld] for p in snakemake.input.cld_exp_data}
ds_ctrl_cld = {p.split("_")[-2]: xr.open_dataset(p)[vars_cld] for p in snakemake.input.cld_ctrl_data}

ds_exp_diag = ds_exp = {p.split("_")[-2]: xr.open_dataset(p).isel(time=slice(1,None))[vars_diag] for p in snakemake.input.diag_exp_data}
ds_ctrl_diag = {p.split("_")[-2]: xr.open_dataset(p)[vars_diag] for p in snakemake.input.diag_ctrl_data}

In [ ]:
def add_diag_to_df(ds_ctrl, ds_exp, var, df_exp, df_ctrl, model,time_unit='year', scale_factor=1):
    dvar_exp = ds_exp[var].mean(dim='time')
    dvar_ctrl = ds_ctrl[var].mean(dim='time')
    dvar_exp = global_avg(dvar_exp).values
    dvar_ctrl = global_avg(dvar_ctrl).values
    if time_unit == 'year':
        tscale = (60*60*24*365)
    elif time_unit == 'day':
        tscale = (60*60*24)
    else:
        tscale = 1

    if var == 'radatm' or var == 'radatmcs':
        lh_cond_water = 2260*(10**3)
        dvar_exp = ((dvar_exp/lh_cond_water)*tscale)*scale_factor
        dvar_ctrl = ((dvar_ctrl/lh_cond_water)*tscale)*scale_factor
    
    df_exp.loc[model,var] = dvar_exp
    df_ctrl.loc[model,var] = dvar_ctrl
    return df_exp, df_ctrl

In [ ]:
df_exp = pd.DataFrame(columns=vars_cld+vars_diag, index=list(ds_exp_cld.keys()))
df_ctrl = pd.DataFrame(columns=vars_cld+vars_diag, index=list(ds_ctrl_cld.keys()))

for model in ds_exp_cld.keys():
    df_exp, df_ctrl = add_diag_to_df(ds_ctrl_cld[model], ds_exp_cld[model], 'pr', df_exp, df_ctrl,model, time_unit=None, scale_factor=1)
    df_exp, df_ctrl = add_diag_to_df(ds_ctrl_cld[model], ds_exp_cld[model], 'clivi', df_exp, df_ctrl,model, time_unit=None, scale_factor=1e3)
    df_exp, df_ctrl = add_diag_to_df(ds_ctrl_diag[model], ds_exp_diag[model], 'radatm', df_exp, df_ctrl,model, time_unit='year', scale_factor=1)
    df_exp, df_ctrl = add_diag_to_df(ds_ctrl_diag[model], ds_exp_diag[model], 'radatmcs', df_exp, df_ctrl,model, time_unit='year', scale_factor=1)
    df_exp, df_ctrl = add_diag_to_df(ds_ctrl_diag[model], ds_exp_diag[model], 'concdust', df_exp, df_ctrl,model, time_unit=None, scale_factor=1)
    df_exp, df_ctrl = add_diag_to_df(ds_ctrl_diag[model], ds_exp_diag[model], 'abs550aer', df_exp, df_ctrl,model, time_unit=None, scale_factor=1)

In [ ]:
df_diff = df_exp - df_ctrl
df_diff['abs550aer_mass_abs'] = df_diff['abs550aer']/(df_diff['concdust']*1e3)
df_diff['abs550aer'] = df_diff['abs550aer']*1e3

In [ ]:
translate_column_names = {
    'abs550aer_mass_abs' : 'DU MAC (m2 g-1)',
    'abs550aer' : 'DU AAOD 550',
    'radatm' : 'ARC (mm year-1)',
    'radatmcs' : 'ARC$_{clearsky}$ (mm year-1)',
    'pr' : 'Precipitation (mm year-1)',
    'clivi' : 'Cloud ice (g m-2)',
}

In [ ]:
vis_df = df_diff.rename(columns=translate_column_names).astype(float)

In [ ]:
context_dict= {
    'axes.grid' : False,
    'axes.spines.right' : False,
    'axes.spines.top' : False,
    'lines.markersize': 8,
}

In [ ]:
colors = get_model_colordict()

In [ ]:
model_order=['EC-Earth3-AerChem','MPI-ESM-1-2-HAM','NorESM2-LM','IPSL-CM6A-LR-INCA','UKESM1-0-LL','CNRM-ESM2-1','GFDL-ESM4','MIROC6','GISS-E2-1-G']
vis_df = vis_df.reindex(model_order)

In [ ]:
with mpl.rc_context(rc=context_dict):
    fig,ax = plt.subplots(ncols=3,figsize=(8.27,2.42)) 

    for index, row in vis_df.iterrows():
        ax[0].scatter(row['ARC (mm year-1)'], row['Precipitation (mm year-1)'], c=colors[row.name], label=row.name,
                      zorder=100)
    for index, row in vis_df.iterrows():
        ax[1].scatter(row['ARC (mm year-1)'], row['ARC$_{clearsky}$ (mm year-1)'], c=colors[row.name], label=row.name,
                      zorder=100)

    for index, row in vis_df.iterrows():
        ax[2].scatter(row['DU AAOD 550'], row['ARC$_{clearsky}$ (mm year-1)'], c=colors[row.name], label=row.name,
                      zorder=100)
    ax[0].set_xlabel('ARC (mm year-1)')
    ax[0].set_ylabel('Precipitation (mm year-1)')
    ax[1].set_xlabel('ARC (mm year-1)')
    ax[1].set_ylabel('ARC$_{clearsky}$ (mm year-1)')
    ax[2].set_xlabel('Dust AAOD 550nm (1*1000)')
    ax[2].set_ylabel('ARC$_{clearsky}$ (mm year-1)')
    slope,intercept, r_val, p_val, std_err = linregress(vis_df['ARC (mm year-1)'], vis_df['Precipitation (mm year-1)'])
    xl = [-12,0]
    yl = [slope*-12+intercept, intercept]
    ax[0].plot(xl, yl, color='red', linestyle='--', linewidth=3, zorder=1)
    eqn = f'y = {slope:.2f}x + {intercept:.2f}'
    ax[0].annotate(eqn, xy=(0.05, 0.95), xycoords='axes fraction', fontsize=10, ha='left', va='top')
    ax[1].set_xlim(-12,2)
    ax[1].set_xticks([-10,-5, 0])
    ax[1].set_ylim(-25, 5)
    
    ax[2].set_xlim(0,3)
    ax[2].set_xticks([0,1, 2, 3])
    ax[2].set_xticklabels([0, 1,2,3])
    ax[2].set_ylim(-25, 5)

    ax[0].set_ylim(-8,2)
    ax[0].set_xlim(-12,1)
    slope,intercept, r_val, p_val, std_err = linregress(vis_df['DU AAOD 550'], vis_df['ARC$_{clearsky}$ (mm year-1)'])
    xl = [0,3]
    yl = [slope*0+intercept, slope*3+intercept]
    ax[2].plot(xl, yl, color='red', linestyle='--', linewidth=3, zorder=1)
    eqn = f'y = {slope:.2f}x + {intercept:.2f}'
    ax[2].annotate(eqn,  xy=(0.2, 0.95), xycoords='axes fraction', fontsize=10, ha='left', va='top', zorder=100)
    plt.subplots_adjust(wspace=0.43)
    handles, labels = ax[1].get_legend_handles_labels()
    fig.legend(handles, labels,loc='upper center', ncol=3, bbox_to_anchor=(0.5,-0.1), frameon=False)
    ax[0].text(x=-0.25, y=1.05, s='a)', fontsize=12, fontweight='bold' ,transform=ax[0].transAxes)

    ax[1].text(x=-0.25, y=1.05, s='b)', fontsize=12, fontweight='bold' ,transform=ax[1].transAxes)
    ax[2].text(x=-0.25, y=1.05, s='c)', fontsize=12, fontweight='bold' ,transform=ax[2].transAxes)
plt.savefig(snakemake.output.outpath, bbox_inches='tight')